In [2]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(r'D:\quant_learning\week4_sql\quant.db')

conn.execute('DROP TABLE IF EXISTS prices')
conn.execute('''CREATE TABLE prices (
    code TEXT, trade_date TEXT, close REAL)''')

rows = [
    ('000001','2024-01-02',11.2),
    ('000001','2024-01-03',11.5),
    ('000001','2024-01-04',11.3),
    ('600519','2024-01-02',1700.0),
    ('600519','2024-01-03',1680.0)]

conn.executemany('INSERT INTO prices VALUES (?,?,?)', rows)
conn.commit()
print('建库完成，当前行数',pd.read_sql('SELECT COUNT(*) AS n FROM prices', conn))


建库完成，当前行数    n
0  5


In [3]:
sql='''
SELECT p.code, p.trade_date, p.close AS today,
    prev.close AS prev_day, ROUND(p.close-prev.close,2) AS chg
FROM prices p
JOIN prices prev
ON p.code=prev.code
AND prev.trade_date=DATE(p.trade_date,'-1 day')
ORDER BY p.code, p.trade_date'''
pd.read_sql(sql, conn)

,code,trade_date,today,prev_day,chg
0,000001,2024-01-03,11.5,11.2,0.3
1,000001,2024-01-04,11.3,11.5,-0.2
2,600519,2024-01-03,1680.0,1700.0,-20.0


In [8]:
sql2='''
SELECT ret, COUNT(*) FROM (SELECT p.code, p.trade_date, p.close AS today,
    prev.close AS prev_day, ROUND(p.close-prev.close,2) AS chg,
    CASE WHEN  ROUND(p.close-prev.close,2)>0 THEN 'up'
    WHEN ROUND(p.close-prev.close,2)=0 THEN 'flat'
    ELSE 'down' END AS ret
FROM prices p
JOIN prices prev
ON p.code=prev.code
AND prev.trade_date=DATE(p.trade_date,'-1 day')
ORDER BY p.code, p.trade_date)
GROUP BY ret
'''
pd.read_sql(sql2, conn)

,ret,COUNT(*)
0,down,2
1,up,1
